# Dunnhumby GraphSAGE · M1~M5_D 8모형 · seed49

**GraphSAGE 기반에서 확정 M2·M3·M4의 독립 효과와 결합 효과를 비교**합니다. 모두 같은 GraphSAGE M1을 기준으로 비교하며 기존 LightGCN/NGCF/GAT 성능·체크포인트는 GraphSAGE 결과로 재사용하지 않습니다.

|순서|모형|M2 표현|M3 그래프|M4 손실|
|---|---|---|---|---|
|1|M1|없음|이진 연결|무가중|
|2|M2|적응형 N/V|이진 연결|무가중|
|3|M3|없음|중심화 N/V|무가중|
|4|M4|없음|이진 연결|고객 내 평균1|
|5|M5_A|적응형 N/V|중심화 N/V|무가중|
|6|M5_B|적응형 N/V|이진 연결|고객 내 평균1|
|7|M5_C|없음|중심화 N/V|고객 내 평균1|
|8|M5_D|적응형 N/V|중심화 N/V|고객 내 평균1|

## 고정 실험조건
- L4 GPU, seed49, train DAY1~704 / test DAY705~711. 신규 상품 추천: 학습 구매쌍을 test 정답·후보에서 제외. MIN_ITEM_INTER=1, uniform 음성1개.
- 모형당 **300epoch 마지막 체크포인트만 test 1회 평가**. validation·holdout·조기종료·중간test·추가시드 자동실행 없음. 결과를 보고 계수·epoch를 바꾸지 않습니다.
- ID64·2층64·batch8192·Adam lr.0005·ID L2 .001·M2 공유 L2 1e-6·GraphSAGE 변환/편향 L2 1e-6. 하나의 optimizer로 ID·GraphSAGE·M2를 공동학습합니다. 사전학습 동결·외부재정렬 없음.
- M2 N/V·아이템 경제 속성·관측량 보조계수·RBF·양성상품의 현재 구매자 제외·학습형 축계수/rho·직교결합은 유지하며 출력64차원입니다.
- M4 고객 내부 구매행 평균1 가중치를 그대로 사용합니다. CLV는 학습구간 최근365일의 historical CLV proxy이며 미래 CLV/증분매출이 아닙니다.

## 이번 GraphSAGE의 정의와 M3 적용
이웃의 평균 표현과 자기 표현을 따로 결합하는 **mean GraphSAGE**입니다. 기존 프로젝트의 GraphSAGE 추천용 설정(ELU·층별 L2 정규화·0/1/2층 평균)을 사용합니다. 원논문의 데이터/분류기 전체 재현은 아닙니다.

`P[u,v] = A_w[u,v] / sum_v A_w[u,v]`

`H_next = L2normalize(ELU(concat(H, P @ H) @ W + b))`

`Z = (E + H1 + H2) / 3; score(u,i) = inner_product(Z_u,Z_i)`

- 각 층은 자기64차원과 이웃64차원을 합친128차원을64차원으로 변환합니다. 두 층을 사용하고 dropout은0입니다. M2가 있으면 이 Z에 확정된 직교 경제표현 결합을 적용한 뒤 추천점수를 계산합니다.
- 이웃 표본추출 없이 전체 이웃을 사용합니다. 자기 표현은 concat으로 들어가므로 인접행렬에 자기연결을 추가하지 않습니다. 이웃 없는 노드의 이웃평균은0이며 자기 경로는 남습니다.
- M3가 없으면 관측 엣지 가중치1의 평균, M3가 있으면 **기존 중심화 N/V 원시 가중치로 계산한 가중평균**입니다. 이미 대칭 정규화된 LightGCN 행렬을 사용하지 않습니다.
- M3 입력·계수는 그대로지만 행별 가중치 합으로 다시 나누기 때문에 공통 행 배율은 상쇄됩니다. 원시 M3에서 점검한 양쪽 총량 성질이 최종 GraphSAGE 전파에서도 그대로 유지된다고 주장하지 않습니다.
- 학습형 ID를 쓰는 추천 실험으로, 학습하지 않은 새 사용자·상품에 대한 귀납 일반화 검증은 아닙니다. 실제 전파/층결합 차이를 protocol에 기록하고 백본별 M1 대비 효과를 비교합니다.

## 저장·재개·시간 확인
- **매epoch** 모델·Adam·numpy/torch/CUDA 난수를 로컬 직렬화 후 Drive에 동기 저장합니다. 이전 checkpoint는 새 파일 복사가 완료된 뒤 교체합니다.
- 끊기면 **같은 링크·코드·참조/출력 경로로 위에서부터 재실행**하세요. 완료모형은 재학습·재평가하지 않고 미완료모형은 마지막 저장 epoch 다음부터 재개합니다. 중간 배치에서 끊기면 해당 미완료 epoch는 다시 실행합니다. Colab 자동 재접속은 제공하지 않습니다.
- 손상 checkpoint/Drive 저장 오류는 파일을 보존하고 중단합니다. 원인을 확인하기 전에 결과 폴더를 삭제하지 마세요.
- 정적 입력은 한 번 준비하되 학습 임베딩은 매 배치 다시 계산합니다. uniform 음성의 동일 표본/RNG를 유지하는 이진검색 검사와 기존 저장 최적화를 사용합니다.
- 첫3epoch의 첫5배치 입력/계산 시간, 매epoch 전체 학습/Drive 저장 시간을 기록합니다. 실제 Dunnhumby/L4 전체 실행시간은 아직 미측정입니다.
- 결과 ZIP: 전체142지표, CLV 세그먼트/노출, 모든 모형간 비교, 진단, 학습이력, 시간, 설정. 미완료 ZIP은 partial로 구분합니다.
- **1시드 결과로 표준편차·95%CI·유의성을 주장하지 않습니다.** 약10시드 최종평균·CLV 배정귀속·새 독립확증을 대체하지 않습니다.

근거: [GraphSAGE 원논문](https://papers.nips.cc/paper/6703-inductive-representation-learning-on-large-graphs.pdf)의 평균 집계·자기/이웃 결합 및 층별 정규화 구조. ELU·층평균·ID/BPR는 프로젝트 추천용 변형으로 별도 명시합니다.
별도 PyG 설치 없이 PyTorch 희소행렬을 사용합니다. 실제 Colab 전체학습은 사용자 실행 단계이며 로컬 합성자료로 수식·개입 경로·재개를 검증했습니다.

같은 출력 폴더를 두 런타임에서 동시에 실행하지 마세요. 평가 후 결과 JSON 저장 전에 끊기면 재접속 시 최종 평가만 재계산될 수 있습니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), '런타임 유형을 L4 GPU로 설정하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU: {GPU}. L4로 변경 후 실행하세요.'
SOURCE_COMMIT = '30d1e2ace42e1697b88649a0d733b616a0582455'
REPO = Path('/content/clv-dh-graphsage-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 확정 CLV 입력과 기존 참조 검증 — 학습 없음
기존 Dunnhumby M2 10시드 및 M4 10시드 폴더에서 원본 보고서의 해시·분할·입력·M2 특징·M3/M4 가중치 정합성을 확인합니다. 파일이 맞지 않으면 학습 전에 중단합니다. **이 단계에서 기존 모델을 새로 학습하지 않습니다.** 재사용 준비 모듈의 A/B 안내가 나타나더라도 실제 실행 목록은 아래 GraphSAGE 8개입니다.

기본 참조 폴더가 이동/변경되었다면 `m2_reference`, `m4_reference`에 보관 원본 ZIP 또는 폴더의 Drive 경로를 넣으세요. 첫 실행 이후에는 경로를 유지하세요. 출력은 별도 GraphSAGE 폴더입니다.

In [ ]:
import clv_dh_graphsage_final8_seed49 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/원본M2결과.zip',
    # m4_reference='/content/drive/MyDrive/clv_m4_user_centered_final10_results.zip',
)
print('M2 입력 검증 참조:', cfg.m2_reference)
print('M3/M4 입력 검증 참조:', cfg.m4_reference)
print('GraphSAGE 새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('실제 새 학습:', list(screen.ORDER), '/ seed49 / 각300epoch')
print('진행·checkpoint·결과 저장:', prepared['run_dir'])

## 2. GraphSAGE 8개 모형 순차 학습
모형당 시작3개 epoch 및 이후5epoch마다 손실·전체 학습 시간·Drive 저장 시간을 표시합니다. 처음5배치가 끝나면 입력/음성추출과 학습계산의 배치당 표본 시간도 먼저 표시하므로 첫epoch가 끝나기 전에도 확인할 수 있습니다.

매epoch checkpoint를 Drive에 저장하고, 각 모형300epoch 완료 시 전체142지표를 평가·저장한 뒤 다음 모형으로 넘어갑니다. **로그 빈도와 무관하게 Drive 저장은 매epoch 실행**됩니다. 학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

각 시간은 다음과 같습니다.
- 입력·음성추출: 같은 순서의 uniform 음성표본 생성과 배치 자료 준비/전송.
- 학습계산: 그래프 전파·BPR 손실·역전파·Adam 업데이트.
- 학습 전체: epoch의 입력·계산·로그/진행기록을 포함한 벽시계 시간.
- Drive 저장완료: 로컬 직렬화·복사/해시·Drive 저장/메타데이터 전체 시간.


In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 시간 기록 확인 — 학습·평가 없음
아래 셀은 저장된 시간 CSV만 읽습니다. 학습 셀이 끝났거나 중단된 뒤 실행하세요. 마지막10개 epoch를 보여주며 전체 기록은 결과 ZIP의 `timing/`에 포함됩니다. 입력 준비 총 시간은 `preparation_timing.json`에 있습니다. 표본 시간은 첫3개 epoch만 있으며, 나머지 빈칸은 미측정입니다.

In [ ]:
import pandas as pd
from IPython.display import display
timing_files = sorted((prepared['run_dir'] / 'progress' / 'timing').glob('*.csv'))
if timing_files:
    timing = pd.concat([pd.read_csv(p) for p in timing_files], ignore_index=True)
    columns = ['model_id', 'epoch', 'train_seconds', 'save_seconds',
               'sample_input_seconds_per_batch', 'sample_compute_seconds_per_batch',
               'checkpoint_local_seconds', 'checkpoint_drive_seconds', 'checkpoint_metadata_seconds']
    display(timing[columns].tail(10))
    print('단위: 초. 표본 계측은 추가 동기화 비용을 포함합니다. test 성능을 측정하지 않습니다.')
else:
    print('아직 완료 epoch의 시간 기록이 없습니다. 초기 배치 시간은 학습 출력에서 확인하세요.')

## 3. 저장된 완료 모형만 비교 — 학습·재평가 없음
8모형 절대지표와 **같은 GraphSAGE M1 대비** 변화율입니다. 미완료 모형은 빈칸이며0점이 아닙니다. 전체142지표·모든 모형 간 비교·개입 진단은 CSV에 보관됩니다. 단일 시드이므로 표준편차/신뢰구간을 만들지 않습니다.

세션이 끊겼다면1번 준비 셀까지 실행한 다음 이 셀로 바로 와도 됩니다. 전체 결과를 받으려면2번 학습 셀을 이어서 실행하세요.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(columns=screen.ORDER))
against_m1 = result['comparison'].query("reference == 'M1'")
if not against_m1.empty:
    display(against_m1.pivot(index='metric', columns='model', values='relative_change_pct')
            .reindex(index=metrics, columns=screen.ORDER[1:]))
    print('GraphSAGE M1 대비 상대변화(%). 모든 비교·불리한 지표도 CSV에 보존합니다.')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
print('seed49 1시드 — 평균/표준편차·유의성·CLV 배정 귀속 판단 아님')

## 4. 결과 ZIP 다운로드
완료 모형의 원본 결과·1~300epoch 학습 이력·전체 비교·설정·입력 검증 기록을 받습니다. 8개 미완료 시 파일명에 `_partial`을 붙입니다. 재개용 대용량 checkpoint는 ZIP에 넣지 않고 **Drive에 그대로 유지**합니다.

In [ ]:
from google.colab import files
import shutil
output = screen.export_zip(prepared, '/content/dh_graphsage_final8_seed49_results.zip')
src = Path(output)
backup = prepared['run_dir'] / src.name
shutil.copy2(src, backup)
assert backup.stat().st_size == src.stat().st_size
print('Drive 보관:', backup, '| ZIP 크기(MB):', round(src.stat().st_size / 1024**2, 3))
print('다운로드가 멈추면 위 Drive 경로에서 직접 다운로드하세요. checkpoint는 Drive에 유지됩니다.')
files.download(str(src))